## Transferring Data from Device manufacturer sites to Influxdb
For flexibility and easy of data manipulation, we transfer the data to influx db.
This will also enable us to have the data downloaded and stored in one place - giving us ownership incase our subscription with the device manufacturer expires.

### Prepare tools for the project
First install the packages and modules necessary to implement the steps of the project.

The documentation for the influxdb_client_3 library can be found [here](https://docs.influxdata.com/influxdb3/cloud-dedicated/reference/client-libraries/v3/python/)

In [3]:
#Install operational packages to help in 
#organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json

#Modules to help interact with influxdb
import os, time
from influxdb_client_3 import InfluxDBClient3, Point


Create a magic function to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [4]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

### Set up influxdb
Prepare the database to receive data from the device manufacturer urls 

In [5]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

#Define some of the arguments required 
#to write the data to the database

token = os.environ.get("INFLUXDB_TOKEN")
org         =   "Air quality - Harrisburg and Philadelphia"
host        =   "https://us-east-1-1.aws.cloud2.influxdata.com"
database    =   "pilot-harrisburg-home"

client = InfluxDBClient3(host=host, token=token, org=org, database=database)

### Upload the data from Devices to influxdb
Make API calls to the device urls and send the data to the influxdb database. Since different manufacturers have different procedures for API calls, we do upload the data in different batches based on the manufacturer

1. QUANTAQ

The guidelines to make API calls to quantaq can be found [here](https://docs.quant-aq.com/software-apis-and-libraries/quantaq-cloud-api). For using python to manipulate data; find the guidance under the [documentation for the py-quantaq](https://quant-aq.github.io/py-quantaq/usage.html#list-all-final-data-for-a-device); which is a quantaq python library for data manipulation. 

In [7]:
#Load the api key and device serial numbers from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")
QUANT001_SN = os.environ.get("QUANT001_SN")
QUANT002_SN = os.environ.get("QUANT002_SN")
CLARITY_APIKEY = os.environ.get("CLARITY_APIKEY")
CLARITY_ORGID = os.environ.get("CLARITY_ORGID")


In [1]:
def get_quantaq_data(url_ending_list):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_ending_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending_list:
        url_additional = url_additional + "/" + adds_on 
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    #Convert to dictionary for easy manipulation in python
    response_dict = json.loads(response_json.content)

    return response_dict

In [9]:
all_devices = ['MOD-X-00959','MOD-X-00958']
json_data = get_quantaq_data(['v1','data','resampled','MOD-X-00959','2025-11-05','2026-01-11','1h'])
#pd.json_normalize(json_data['data'])
json_data


{'error': 'not found'}

In [10]:

json_data = get_quantaq_data(['v1','devices','MOD-X-00959','data'])


In [ ]:
complete_url = 'https://api.quant-aq.com/v1/data/resampled/MOD-X-00959/2025-11-05/2026-01-11/1h/'

complete_url2 = 'https://api.quant-aq.com/v1/data/most-recent/MOD-X-00959'
complete_url3 = 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/'
response_json = requests.get(complete_url3, auth=(QUANTAQ_APIKEY,""))

#Convert to dictionary for easy manipulation in python
response_data = json.loads(response_json.content)

In [ ]:
complete_url4 = 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data//'
response_json = requests.get(complete_url4, auth=(QUANTAQ_APIKEY,""))
json.loads(response_json.content)

{'error': 'not found'}

In [33]:
response_data

{'data': [{'co': 175.99,
   'geo': {'lat': 40.3156, 'lon': -76.8962},
   'met': {'wx_dew_point': 0.0,
    'wx_pressure': 0.0,
    'wx_rh': 0.0,
    'wx_temp': 0.0,
    'wx_u': None,
    'wx_v': 0.0,
    'wx_wd': 0.0,
    'wx_ws': 0.0,
    'wx_ws_scalar': 0.0},
   'model': {'gas': {'co': 20508, 'no': 20509, 'no2': 20510, 'o3': 20511},
    'pm': {'pm1': 18867, 'pm10': 18869, 'pm25': 18868}},
   'no': 2.43,
   'no2': 6.67,
   'o3': 38.83,
   'pm1': 8.52,
   'pm10': 11.29,
   'pm25': 8.78,
   'raw_data_id': 23436284,
   'sn': 'MOD-X-00959',
   'timestamp': '2026-02-04T17:43:45',
   'timestamp_local': '2026-02-04T12:43:45',
   'url': 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/23436218'},
  {'co': 181.54,
   'geo': {'lat': 40.3156, 'lon': -76.8962},
   'met': {'wx_dew_point': 0.0,
    'wx_pressure': 0.0,
    'wx_rh': 0.0,
    'wx_temp': 0.0,
    'wx_u': None,
    'wx_v': 0.0,
    'wx_wd': 0.0,
    'wx_ws': 0.0,
    'wx_ws_scalar': 0.0},
   'model': {'gas': {'co': 20508, 'no': 2050

In [16]:
pd.json_normalize(response_data['data'])

KeyError: 'data'

In [8]:
pd.json_normalize(json_data['data'])


,co,no,no2,o3,pm1,pm10,pm25,raw_data_id,sn,timestamp,...,met.wx_wd,met.wx_ws,met.wx_ws_scalar,model.gas.co,model.gas.no,model.gas.no2,model.gas.o3,model.pm.pm1,model.pm.pm10,model.pm.pm25
0,204.29,0.63,5.69,28.37,4.01,6.79,4.13,21591766,MOD-X-00959,2026-01-16T20:30:44,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
1,210.79,1.31,6.07,29.20,4.14,6.07,4.35,21591765,MOD-X-00959,2026-01-16T20:29:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
2,206.82,1.98,5.43,28.96,3.65,15.23,3.86,21591764,MOD-X-00959,2026-01-16T20:28:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
3,208.46,1.31,5.43,28.96,3.50,3.91,3.70,21591763,MOD-X-00959,2026-01-16T20:27:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
4,209.41,0.80,8.06,26.24,3.99,4.83,4.19,21591762,MOD-X-00959,2026-01-16T20:26:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
5,207.58,0.97,5.79,28.88,4.53,5.05,4.63,21591415,MOD-X-00959,2026-01-16T20:25:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
6,204.78,2.32,5.81,28.43,3.87,4.61,4.01,21591414,MOD-X-00959,2026-01-16T20:24:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
7,212.01,0.62,6.71,28.00,4.77,5.69,4.87,21591416,MOD-X-00959,2026-01-16T20:23:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
8,205.39,1.81,6.06,27.83,4.78,6.15,4.88,21591413,MOD-X-00959,2026-01-16T20:22:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
9,205.64,0.97,5.63,28.80,4.25,4.58,4.34,21591417,MOD-X-00959,2026-01-16T20:21:43,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868


Using the py-QuantAQ library to get data from the web and manipulate it 


In [9]:
%%skip
pip install py-quantaq

In [10]:
#Bring in the the quantaq libray and instantiate the client object
from quantaq.utils import to_dataframe
from quantaq import QuantAQAPIClient

client_quantaq = QuantAQAPIClient(api_key=QUANTAQ_APIKEY)

Get all the data from the commissioning date to now. 
This command can be performed only once and then the database will be updated regularly thereafter.

In [12]:
%%skip
#Define the devices form which the data will be extracted 
quantaq_devices = ['MOD-X-00959','MOD-X-00958']

#Initialize the dataframe to be used to make a 
# singular table to save to the influxdb database
quantaq_all_data_df = []

#Collect the date all the devices from commissioning date to jan-13-2026
for device in quantaq_devices:
    for each in pd.date_range(start='2025-11-04', end='2026-01-13'):
        quantaq_all_data_df.append(
            to_dataframe(client_quantaq.data.bydate(sn=device, date=str(each.date())))
        )
quantaq_all_data_df = pd.concat(quantaq_all_data_df)

Store the file as pickle so as to avoid length periods of downloading next time

In [12]:
%%skip
quantaq_all_data_df.to_pickle('quantaq_all_devices_2025_11_to_2026_01')

In [13]:
quant_all_to_01_12_2026 = pd.read_pickle('quantaq_all_devices_2025_11_to_2026_01')
quant_all_to_01_12_2026

,co,no,no2,o3,pm1,pm10,pm25,raw_data_id,sn,timestamp,...,met.wx_wd,met.wx_ws,met.wx_ws_scalar,model.gas.co,model.gas.no,model.gas.no2,model.gas.o3,model.pm.pm1,model.pm.pm10,model.pm.pm25
0,38.26,3.48,5.57,7.75,8.67,44.64,9.62,13608141,MOD-X-00959,2025-11-05 23:59:06,...,0.0,0.0,0.0,18903.0,18917.0,18934.0,18935.0,18867,18869,18868
1,29.20,4.00,5.30,8.02,7.98,40.72,8.83,13607850,MOD-X-00959,2025-11-05 23:58:06,...,0.0,0.0,0.0,18903.0,18917.0,18934.0,18935.0,18867,18869,18868
2,30.11,3.45,5.77,8.10,8.60,36.28,9.22,13607849,MOD-X-00959,2025-11-05 23:57:06,...,0.0,0.0,0.0,18903.0,18917.0,18934.0,18935.0,18867,18869,18868
3,27.40,3.99,5.96,7.51,8.31,22.55,9.26,13607848,MOD-X-00959,2025-11-05 23:56:06,...,0.0,0.0,0.0,18903.0,18917.0,18934.0,18935.0,18867,18869,18868
4,32.55,3.27,5.82,8.10,9.02,31.98,9.65,13607845,MOD-X-00959,2025-11-05 23:55:06,...,0.0,0.0,0.0,18903.0,18917.0,18934.0,18935.0,18867,18869,18868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1050,231.20,1.00,16.91,36.18,1.54,2.40,1.58,21120552,MOD-X-00958,2026-01-12 00:04:46,...,NaN,NaN,NaN,20450.0,20451.0,18932.0,18933.0,18864,18866,18865
1051,227.44,0.84,17.57,33.76,1.33,2.30,1.40,21120548,MOD-X-00958,2026-01-12 00:03:46,...,NaN,NaN,NaN,20450.0,20451.0,18932.0,18933.0,18864,18866,18865
1052,224.70,-0.02,16.90,34.77,1.08,1.27,1.21,21120549,MOD-X-00958,2026-01-12 00:02:46,...,NaN,NaN,NaN,20450.0,20451.0,18932.0,18933.0,18864,18866,18865
1053,222.11,0.11,16.91,35.71,1.10,2.01,1.17,21120551,MOD-X-00958,2026-01-12 00:01:46,...,NaN,NaN,NaN,20450.0,20451.0,18932.0,18933.0,18864,18866,18865


In [ ]:
def upload_to_inlfuxdb(data_frame_to_save, measurement_name, 
                       index_field, client_name, tag_cols_list):
    """
    A function that takes the name of the measurement(table) 
    and one field name to serve as the index and wites them into
    a defined database on influxdb

    Args:
        data_frame_to_save(df)  : The dataframe to be sent to the database
        measurement_name(string) : The measurement or table name to store the dataset
        index_field(string)     : A timefield from the measurement to be used as an index
                                This also serves as the timeseries reference column
        client_name(object)     : The influxdb object with a defined database 
        tag_cols_list (list)    : A list of column names to act as tags

    Return : 
            Prints that it completed the task
    """ 

    #Make the index field to be the index so that it acts as the timestamp
    data_frame_to_save_index = data_frame_to_save.set_index(index_field)
    
    #Write to the database and the let user know that it is written
    client_name.write(record=data_frame_to_save_index, 
                      data_frame_measurement_name=measurement_name,
                      data_frame_tag_columns=tag_cols_list)
    
    print("Done! Check your influxdb to confirm!")

Transform the QuantaAQ data obtained through the API into a datframe, and send it to the influxdb database for further querrying and vizualization

In [15]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data','?per_page=1000'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.json_normalize(data=data_quant002_dict)

#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=data_quant002_df, 
                   measurement_name='quantaq-pilot-tags-clean', 
                   index_field='timestamp', 
                   client_name=client,
                   tag_cols_list=['sn','geo.lat','geo.lon'])

Done! Check your influxdb to confirm!


Save the entrie dataset from commissioning date 

#### 2. Clarity
The general guidelines for accessing clarity data through the api can be found [here](https://api-guide.clarity.io/); These guideline use examples based on POSTMAN - a webtool for obtaining data using APIs. Clarity has a python library for retrieving data found [here](https://github.com/a2gov/clarityio). Here we use the python library.

First install the clarity library

In [16]:
%%skip
pip install clarityio

Make initial connections with the clarity library


In [17]:
import clarityio # Offers methods to connect to the clarity api
from io import StringIO  # Helps to read file as csv and transform to df
api_connection = clarityio.ClarityAPIConnection(api_key=CLARITY_APIKEY, org=CLARITY_ORGID)

Getting the data from the APi before uploading to the database

In [18]:
def create_df_from_date(start_time):

    """
    A function that produces a clarity dataframe from a given to now

    Args:
        start_time(string) : Date and time from to start pulling records
                             In the format of '2026-01-04T00:00:00Z' 
    
    Return:
        latest_clarity_df (dataframe) : A dataframe with data from the given date to now

    """

    #Attemopt to get data from a certain date
    request_body_pilot = {
        'allDatasources': True,
        'outputFrequency': 'hour',
        'format':       'csv-wide',
        'startTime': start_time
        }

    #get the data in form of csv, ocnvert it to a dataframe and 
    # for easy manipulation and saving to the database
    response_wide2 = api_connection.get_recent_measurements(data=request_body_pilot)
    clarity_df = pd.read_csv(StringIO(response_wide2), sep=",")

    #Ensure that only meaningful columns exist to allow 
    # easy data search and manipulation in influxdb
    latest_clarity_df = clarity_df.dropna(axis='columns', how='all')

    return latest_clarity_df


Upload the data from commissioning date to now. This will be the basis from which to contineously update the data

In [ ]:

commissioning_time = '2026-01-04T01:00:00Z'
commission_to_jan_26_df= create_df_from_date(start_time=commissioning_time)

In [20]:
commission_to_jan_26_df

,datasourceId,sourceId,sourceType,outputFrequency,startOfPeriod,endOfPeriod,locationLatitude,locationLongitude,no2Conc1HourMean.raw,no2Conc1HourMean.status,...,pm2_5ConcNum1HourMean.value,pm2_5ConcNum24HourRollingMean.raw,pm2_5ConcNum24HourRollingMean.status,pm2_5ConcNum24HourRollingMean.value,relHumidInternal1HourMean.raw,relHumidInternal1HourMean.status,relHumidInternal1HourMean.value,temperatureInternal1HourMean.raw,temperatureInternal1HourMean.status,temperatureInternal1HourMean.value
0,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-13T15:00:00Z,2026-01-13T16:00:00Z,40.315627,-76.895969,5.14,calibration-missing,...,15.31,20.34,sensor-ready,20.34,55.47,sensor-ready,55.47,4.56,sensor-ready,4.56
1,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-13T15:00:00Z,2026-01-13T16:00:00Z,40.264301,-76.851220,1.04,calibration-missing,...,10.21,12.04,sensor-ready,12.04,44.97,sensor-ready,44.97,6.87,sensor-ready,6.87
2,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-13T15:00:00Z,2026-01-13T16:00:00Z,40.315625,-76.895981,16.42,calibration-missing,...,15.34,19.37,sensor-ready,19.37,54.39,sensor-ready,54.39,4.50,sensor-ready,4.50
3,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-13T14:00:00Z,2026-01-13T15:00:00Z,40.315627,-76.895969,28.39,calibration-missing,...,23.07,20.19,sensor-ready,20.19,68.29,sensor-ready,68.29,-0.05,sensor-ready,-0.05
4,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-13T14:00:00Z,2026-01-13T15:00:00Z,40.264301,-76.851220,8.90,calibration-missing,...,11.78,12.07,sensor-ready,12.07,55.43,sensor-ready,55.43,2.09,sensor-ready,2.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-10T16:00:00Z,2026-01-10T17:00:00Z,40.264301,-76.851220,13.16,calibration-missing,...,13.01,27.92,sensor-ready,27.92,74.44,sensor-ready,74.44,6.21,sensor-ready,6.21
215,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-10T16:00:00Z,2026-01-10T17:00:00Z,40.315625,-76.895981,13.60,calibration-missing,...,13.33,34.38,sensor-ready,34.38,73.82,sensor-ready,73.82,6.82,sensor-ready,6.82
216,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-10T15:00:00Z,2026-01-10T16:00:00Z,40.315627,-76.895969,10.66,calibration-missing,...,11.92,38.95,sensor-ready,38.95,74.66,sensor-ready,74.66,6.35,sensor-ready,6.35
217,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-10T15:00:00Z,2026-01-10T16:00:00Z,40.264301,-76.851220,11.03,calibration-missing,...,12.77,28.64,sensor-ready,28.64,73.80,sensor-ready,73.80,6.36,sensor-ready,6.36


In [21]:
commission_to_jan_26_df[['sourceId','startOfPeriod','endOfPeriod']].sort_values(by='startOfPeriod', ascending=True)

,sourceId,startOfPeriod,endOfPeriod
218,A93NM49L,2026-01-10T15:00:00Z,2026-01-10T16:00:00Z
217,A47HWW6V,2026-01-10T15:00:00Z,2026-01-10T16:00:00Z
216,A44MFTF3,2026-01-10T15:00:00Z,2026-01-10T16:00:00Z
213,A44MFTF3,2026-01-10T16:00:00Z,2026-01-10T17:00:00Z
215,A93NM49L,2026-01-10T16:00:00Z,2026-01-10T17:00:00Z
...,...,...,...
4,A47HWW6V,2026-01-13T14:00:00Z,2026-01-13T15:00:00Z
3,A44MFTF3,2026-01-13T14:00:00Z,2026-01-13T15:00:00Z
2,A93NM49L,2026-01-13T15:00:00Z,2026-01-13T16:00:00Z
1,A47HWW6V,2026-01-13T15:00:00Z,2026-01-13T16:00:00Z


Use the influxdb query package to get the latest date from the database. Look at the date and ensure that it makes sense. 

In [22]:
query_date = """SELECT *
                FROM  'clarity-pilot-tags' 
                ORDER BY time DESC LIMIT 1
             """
    
#Use the queery on the influxdb database
table = client.query(query=query_date, language='sql')

#Obtain the date to use as a start date in the uplaoding 
# the most recent data from clarity to influxdb
most_recent_clarity_df = table.to_pandas()
latest_date = most_recent_clarity_df.startOfPeriod[0]

#verify that the date is in the right format and makes sense
print(latest_date)


2026-01-09T20:00:00Z


From the clarity API, get the data from the most recent date of saving in influxdb to now. And then make a dataframe from it which will later be added to the influxdb database

In [23]:
latest_clarity_df = create_df_from_date(start_time=latest_date)

In [24]:
#Sample for this week (01/04/25 - 01/09/25)
week_df = create_df_from_date(start_time="2026-01-04T00:00:00Z")

Upload this dataframe to the influxdb database in the cloud. Ensure that the measurement name and client name in the uplaod function match with what was used to generate the dataframe

In [25]:
#Define the columns to use as tags so that datapoints 
# recorded at exactly the same time can be recorded
columns_tags = ['datasourceId', 'locationLatitude',
                'locationLongitude','sourceId']
#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=week_df, 
                   measurement_name='clarity-pilot-tags', 
                   index_field='endOfPeriod', 
                   client_name=client,
                   tag_cols_list=columns_tags)

Done! Check your influxdb to confirm!


Since the create_df_from_date function can only get data for a short time (24 hours for minute frequencies). We can instead retrieve large chucks of data using reports and then use the create_df_from_date for small amaounts of data

In [ ]:
from time import sleep
import requests

def request_and_fetch_a_report(start_time, end_time):

    """
    A function that retrieves a one-minute-frequency 
    report in csv-wide format for all devices from clarity 
    A maximum of 30 reports can be retrieved in 24 hours

    Args:
        start_time(string) : Starting time for the period of the report 
                             the format of "yyyy-mm-ddThh:mm:ss.sssZ"
        end_time(string)   : End time for the period of the report in 
                             the format of "yyyy-mm-ddThh:mm:ss.sssZ"

    Return:
        clarity_report_df(DataFrame) : A dataframe report with the clarity 
                                        data in the specified timeframe
    """

    #Define the base part of the clarity url to be used completed in subsequent requests 
    clarity_api_base_url = 'https://clarity-data-api.clarity.io'

    headers = {"x-api-key": CLARITY_APIKEY}

    # request the report
    result = requests.post(url=clarity_api_base_url + "/v2/report-requests",
                           headers=headers,
                           json={
                               "org": CLARITY_ORGID,
                               "report": "datasource-measurements",
                               "allDatasources": True,
                               "outputFrequency": "minute",
                               "startTime": start_time,
                               "endTime": end_time,
                           })
    result.raise_for_status()
    result_json = result.json()
    reportId = result_json['reportId']

    # poll for its completion
    for i in range(12):
        print("sleeping 1 minute")
        sleep(60)
        print("fetching report status ... ", end="")
        statusUrl = clarity_api_base_url + f"/v2/report-requests/{reportId}"
        result = requests.get(url=statusUrl, headers=headers)
        result.raise_for_status()
        result_json = result.json()
        print(result_json.get("reportStatus"))
        if result_json.get("reportStatus") != 'in-progress':
            break

    print(result_json)

    #Define a variable to store the csv file
    filename = None

    if result_json.get("reportStatus") == 'succeeded':
        # if it succeeded, fetch the resulting files
        for i, url in enumerate(result_json['urls']):
            with requests.get(url=url, stream=True) as result:
                result.raise_for_status()
                filename = f"extract_{i}.csv"
                # stream to disk
                with open(f"{filename}", "w") as f:
                    for chunk in result.iter_content(1024 * 1024, decode_unicode=True):
                        f.write(chunk)
    
    #Remove all null columns to reduce redudancy
    report_df = pd.read_csv(filename)
    
    clarity_report_df = report_df.dropna(axis='columns', how="all")


    return clarity_report_df


Use the request_and_fetch_a_report function to get the report from clarity and then uplaod it to inlfuxdb.

In [ ]:
clarity_from_comm_df = request_and_fetch_a_report(start_time="2025-11-01T00:00:00.000Z",
                                                    end_time="2026-01-15T00:00:00.000Z")

In [68]:
#Upload the report to influxdb
upload_to_inlfuxdb(data_frame_to_save=clarity_from_comm_df,
                   measurement_name='clarity-pilot-alldata-tags',
                   index_field='time',
                   client_name=client,
                   tag_cols_list=['datasourceId','sourceId','sourceType',
                                  'locationLatitude','locationLongitude']
                   )
clarity_from_comm_df

ApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Tue, 13 Jan 2026 20:57:08 GMT', 'Content-Type': 'application/json', 'Content-Length': '18189', 'Connection': 'keep-alive', 'influxdata-partial-write': 'true', 'trace-id': '5ae253a60f22879a', 'trace-sampled': 'false', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'X-Influxdb-Request-ID': '753466882426ce9e429bc9d307d5a527', 'X-Influxdb-Build': 'Cloud'})
HTTP response body: {"code":"invalid","message":"partial write has occurred, errors encountered on line(s): line 1: observed timestamp 2025-11-07T18:08:37.537+00:00 is outside of the retention period for the namespace: minimum acceptable timestamp is 2025-12-14T20:57:07.853076198+00:00\nline 2: observed timestamp 2025-11-07T18:13:00.180+00:00 is outside of the retention period for the namespace: minimum acceptable timestamp is 2025-12-14T20:57:07.853076198+00:00\nline 3: observed timestamp 2025-11-07T18:25:52.700+00:00 is outside 

In [67]:
clarity_from_comm_df.columns

Index(['datasourceId', 'sourceId', 'sourceType', 'outputFrequency', 'time',
       'locationLatitude', 'locationLongitude', 'no2ConcIndividual.raw',
       'no2ConcIndividual.status', 'no2ConcIndividual.value',
       'o3ConcIndividual.raw', 'o3ConcIndividual.status',
       'pm10ConcMassIndividual.raw', 'pm10ConcMassIndividual.status',
       'pm10ConcNumIndividual.raw', 'pm10ConcNumIndividual.status',
       'pm10ConcNumIndividual.value', 'pm1ConcMassIndividual.raw',
       'pm1ConcMassIndividual.status', 'pm1ConcNumIndividual.raw',
       'pm1ConcNumIndividual.status', 'pm1ConcNumIndividual.value',
       'pm2_5ConcMassIndividual.raw', 'pm2_5ConcMassIndividual.status',
       'pm2_5ConcMassIndividual.value', 'pm2_5ConcNumIndividual.raw',
       'pm2_5ConcNumIndividual.status', 'pm2_5ConcNumIndividual.value',
       'relHumidInternalIndividual.raw', 'relHumidInternalIndividual.status',
       'relHumidInternalIndividual.value', 'temperatureInternalIndividual.raw',
       'temperature

#### 2. AirGradient

In [ ]:
query_date = """SELECT *
                FROM  'airgradient-pilot-hour' 
                ORDER BY time DESC LIMIT 1
             """
    
#Use the queery on the influxdb database
table = client.query(query=query_date, language='sql')

#Obtain the date to use as a start date in the uplaoding 
# the most recent data from airgradient to influxdb
most_recent_airgradient_df = table.to_pandas()
latest_date_airgradient_hr = str(most_recent_airgradient_df.
                              time[0])[:10].replace('-','')

#Use two days from now as the end date - for the purposes of buffering
end_date_airgradient_hr = f'{date.today() + timedelta(days=2)}'[:10].replace('-','')

#verify that the date is in the right format and makes sense
print(f'start: {latest_date_airgradient_hr }')
print(f'end: {end_date_airgradient_hr}')